In [28]:
from comptox_ai.db import GraphDB
import pandas as pd

In [39]:
db = GraphDB(hostname="neo4j.comptox.ai")
def make_qsar_dataset(chemical_list, assay_abbreviation):
    querystr = """
    MATCH (cl:ChemicalList {{ listAcronym: "{0}" }})-[r1:LISTINCLUDESCHEMICAL]->(c:Chemical)-[r2]->(a:Assay {{ commonName: "{1}" }})
    RETURN 
        c.commonName AS chemical,
        c.sMILES AS smiles,
        c.maccs AS maccs,
        CASE TYPE(r2)
          WHEN 'CHEMICALHASINACTIVEASSAY' THEN 0
          WHEN 'CHEMICALHASACTIVEASSAY' THEN 1
          ELSE 2
        END as active;
    """
    
    res = db.run_cypher(querystr.format(chemical_list, assay_abbreviation))
    print("INFO: Number of chemicals in dataset:", len(res))
    if len(res) < 500:
        print("This might not be enough chemicals to train a model - you should consider generating a different dataset instead.")
    print()

    return pd.DataFrame(res)

In [40]:
# Example of how to run the function:
dataset = make_qsar_dataset("GHSEYESKIN", "tox21-er-bla-agonist-p2")

# Showing the dataset that results from it:
dataset

INFO: Number of chemicals in dataset: 3078



,chemical,smiles,maccs,active
0,2-Amino-5-nitrophenol,NC1=C(O)C=C(C=C1)[N+]([O-])=O,0000000000000000000000010000000000000000000000...,0
1,Acrylonitrile,C=CC#N,0000000000000000000000000000000001000000100000...,0
2,Actinomycin D,[H][C@@]12CCCN1C(=O)[C@H](NC(=O)[C@@H](NC(=O)C...,0000000000000000000000000000000000000000000000...,0
3,4-Acetylaminophenylacetic acid,CC(=O)NC1=CC=C(CC(O)=O)C=C1,0000000000000000000000000000000000000000000000...,0
4,Acrylamide,NC(=O)C=C,0000000000000000000000000000000001000000000000...,0
...,...,...,...,...
3073,"3,3'-Dimethylbisphenol A",CC1=CC(=CC=C1O)C(C)(C)C1=CC(C)=C(O)C=C1,0000000000000000000000000000000000000000000000...,1
3074,10-Chloro-9-anthraldehyde,ClC1=C2C=CC=CC2=C(C=O)C2=CC=CC=C12,0000000000000000000000000000000000000000000000...,1
3075,Dicyclopentamethylenethiuram disulfide,S=C(SSC(=S)N1CCCCC1)N1CCCCC1,0000000000000100000000000000000000000000000000...,1
3076,Pamoic acid,OC(=O)C1=C(O)C(CC2=C3C=CC=CC3=CC(C(O)=O)=C2O)=...,0000000000000000000000000000000000000000000000...,1
